# Naive buffered async (notebook 04)

Collect `buffer_size` updates, then **replace** the global model with their average.

**How it works**

1. Clients train continuously, as in async FL.
2. Arriving updates wait in a buffer.
3. When the buffer is full: new global model = FedAvg of the buffered **client models**.

> **Caution:** this ignores the current global model and staleness completely. A buffer full of stale models can undo recent progress. It's a useful *naive* baseline.

## 1. Load the simulator

Gives this notebook everything from `simulator.ipynb`: the settings, `Approach`, `fedavg`, `apply_delta` and `run`.

In [1]:
%run ../simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class BufferedFedAvg(Approach):
    name = "buffered_fedavg"
    label = "Async staleness unweighted K=5"          # shown in plot legends
    buffer_size = 5

    def start(self, server):
        self.buffer = []
        super().start(server)     # default start: every client trains

    def on_update(self, server, update):
        self.buffer.append(update)
        if len(self.buffer) == self.buffer_size:
            server.set_model(fedavg(self.buffer), self.buffer)
            self.buffer = []

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(BufferedFedAvg(buffer_size=3))`. It is saved as a separate run.

In [3]:
results = run(BufferedFedAvg())

buffered_fedavg              seed=42   val= 93.26%  acc= 94.65%  time=   560.2s  updates=  100  worst10%= 93.51%  wall=5.6s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
20,42,560.1644,100,0.9465,0.1797,0.9351,2.8
